# Yelp review labeling — eight qualifiers

This notebook preserves `gpt-4.1-mini` and the eight `ReviewLabels` fields. Setup never submits requests.

**Concurrent:** useful for a small test or when you want results immediately. Up to five requests can run together. Concurrency reduces waiting, not the token price.

**Batch:** recommended for the large offline run if waiting up to 24 hours is acceptable. OpenAI documents a 50% discount versus synchronous processing and separate limits. Preparation is local; submission is a separate, guarded cell.

Start with **50 reviews**. Finish independent holdout validation before using labels as validated features. This notebook does not alter the source dataset or human labels.

Run **C1–C4** for setup. Then choose **C5** (concurrent), or **B1 → B2 → B3 → B4** (Batch). Run **R1** for usage/counts and **R2** for a CSV export. Do not use Run All to start work; action flags default to False.

The original notebook is preserved as `llm_data_run.sequential_backup.ipynb`. It is historical: do not run it alongside this notebook.


## C1 — Install dependencies (once)
This installs packages, not paid API requests. Restart the kernel if package updates require it.

In [ ]:
%pip install -U openai pydantic python-dotenv pandas


## C2 — Settings and imports (no API calls)
`RESULTS_FILE` selects the output to resume. The default is a new file for the current eight-label prompt. The older sequential file contains 9,825 results with a DIFFERENT prompt hash. To resume that file, restore its exact original prompt and `full_v1_8qualifiers` version, then select its filename. The helper will reject any mismatch; never override the stored hash just to make it pass.

`RUN_LIMIT = 50` means at most 50 **unfinished** reviews per action. Use `None` only when ready for all remaining reviews. Keep the prompt and CSV unchanged during a run.

Set RPM and TPM budgets below the limits shown for your project/model in the OpenAI dashboard. These defaults are conservative; five simultaneous requests are not a guarantee of five times the speed. The token limiter uses a deliberately high UTF-8 byte estimate plus output allowance, not a billing estimate.

Prices below are configurable **USD per million tokens**, not cost per review. Verify your account pricing before a large run. Cached input is a subset of input tokens. Batch cached pricing is an explicit estimate assumption; edit it if your billing terms differ.


In [18]:
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI

PROJECT_DIR = Path('/Users/rejaimes/repos/coffee-project')
DATA_DIR = PROJECT_DIR / 'data' / 'processed'
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from labeling_runner import (
    Run, ReviewLabels, QUALIFIERS, MODEL, run_concurrent,
    prepare_batch, submit_batch, batch_status, collect_batch,
    recover_batch, abandon_prepared, usage_report,
)

PROMPT_PATH = PROJECT_DIR / 'src' / 'prompt_v3.txt'
PROMPT_VERSION = 'full_v3_8qualifiers'
RESULTS_FILE = DATA_DIR / 'llm_labels_full_v3_8qualifiers.jsonl'
# Older sequential file (requires its original matching prompt/version):
# RESULTS_FILE = DATA_DIR / 'llm_labels_full_v1_8qualifiers_20261022_171101.jsonl'
BATCH_ROOT = DATA_DIR / 'labeling_batches'
# To reopen a batch after restarting, paste its printed manifest path here.
MANIFEST_PATH = None

RUN_LIMIT = None
CONCURRENCY = 5              # 1–5
REQUESTS_PER_MINUTE = 60     # Adjust to YOUR account limits
TOKENS_PER_MINUTE = 30000    # Adjust to YOUR account limits
MAX_RETRIES = 4              # Additional attempts for transient errors only
MAX_OUTPUT_TOKENS = 512      # Incomplete responses are failures, never labels
RETRY_FAILED_ONLY = False    # True = select unresolved failures only
REPAIR_INCOMPLETE_TAIL = False

PRICING_USD_PER_MILLION = {
    'concurrent': {'input': 0.40, 'cached_input': 0.10, 'output': 1.60},
    'batch': {'input': 0.20, 'cached_input': 0.05, 'output': 0.80},
}

# Key locations are explicit; no key is displayed or stored in this notebook.
load_dotenv(PROJECT_DIR / 'src' / '.env')
load_dotenv(DATA_DIR / '.env')
PROMPT = PROMPT_PATH.read_text(encoding='utf-8').strip()
assert PROMPT, 'Prompt is empty.'
if 'social_setting' in PROMPT or 'nine labels' in PROMPT:
    print('Prompt still mentions nine/social_setting. Output schema enforces eight. '
          'See prompt review below; do not silently change a prompt used by existing results.')


## C3 — Read source data (local only)
Blank reviews are excluded and displayed for inspection, not silently labeled as zero. No source files are rewritten.

In [19]:
df_reviews = pd.read_csv(DATA_DIR / 'santa_barbara_data.csv', low_memory=False)
df_run = df_reviews[['review_id', 'business_id', 'text']].copy()
assert df_run['review_id'].notna().all(), 'Missing review IDs.'
assert df_run['review_id'].is_unique, 'Duplicate review IDs.'
blank_text = df_run['text'].fillna('').str.strip().eq('')
blank_reviews = df_run.loc[blank_text].copy()
df_run = df_run.loc[~blank_text].copy()
print(f'Source: {len(df_reviews):,}; eligible: {len(df_run):,}; blank: {len(blank_reviews):,}')


Source: 27,128; eligible: 27,128; blank: 0


## C4 — Validate selected results and preview pending work (local only)
Only compatible, validated successful IDs are skipped. Legacy sequential records have no schema hash, so their exact eight field names and integer values are validated directly. Their original model, prompt hash/version, IDs, and dates must match. Legacy text hashes and usage cannot be reconstructed reliably; keep the source unchanged.

If interrupted JSONL has an unfinished last line, this cell stops. Set `REPAIR_INCOMPLETE_TAIL=True` in C2 and rerun C2–C4 to explicitly repair it. The helper first makes a byte-for-byte backup, quarantines only unfinished final bytes, and retains all valid records. A valid final record without a newline is kept. Malformed complete lines, duplicate IDs, or incompatible records are rejected without repair.

One local lock prevents simultaneous writers. Do not run the old sequential notebook in parallel. Active/prepared batches reserve their review IDs so concurrent mode will not repeat them.


In [20]:
run = Run(
    df_run.to_dict('records'), PROMPT, PROMPT_VERSION,
    RESULTS_FILE, BATCH_ROOT, repair_tail=REPAIR_INCOMPLETE_TAIL,
)
print(run.counts())
print('Selected for next action:', len(run.pending(RUN_LIMIT, RETRY_FAILED_ONLY)))
print('Successes:', run.path)
print('Failures:', run.fail_path)
print('Existing local batches:')
for path, manifest in run.manifests():
    print(manifest['state'], manifest.get('batch_id'), path)


{'completed': 50, 'remaining': 27078, 'failed_unresolved': 0, 'reserved_for_batch': 0}
Selected for next action: 27078
Successes: /Users/rejaimes/repos/coffee-project/data/processed/llm_labels_full_v3_8qualifiers.jsonl
Failures: /Users/rejaimes/repos/coffee-project/data/processed/llm_labels_full_v3_8qualifiers.failures.jsonl
Existing local batches:


## C5 — Start concurrent requests (paid action, explicitly enabled)
Change the flag to True **only in this cell** when ready. It resets before starting. A fixed worker pool makes at most five independent requests; one coordinator writes successes as they finish. Shared RPM/TPM limits, exponential backoff with jitter, and server Retry-After cooldowns limit pressure on the API. SDK automatic retries are disabled to keep the retry budget bounded.

To stop: interrupt this cell. Already saved results remain. A request that reached the server but was interrupted before saving may still be billed. Rerun C4 and C5 to resume. Failures are logged separately and never count as success; `RETRY_FAILED_ONLY=True` selects them. Authentication/quota failures stop scheduling more work.


In [ ]:
START_CONCURRENT = True
if START_CONCURRENT:
    START_CONCURRENT = False
    async with AsyncOpenAI(max_retries=0, timeout=90.0) as async_client:
        await run_concurrent(
            run, async_client, limit=RUN_LIMIT, concurrency=CONCURRENCY,
            rpm=REQUESTS_PER_MINUTE, tpm=TOKENS_PER_MINUTE,
            max_retries=MAX_RETRIES, max_output_tokens=MAX_OUTPUT_TOKENS,
            retry_failed_only=RETRY_FAILED_ONLY,
        )
else:
    print('Not started. Set START_CONCURRENT=True in this cell when ready.')


Starting: {'completed': 50, 'remaining': 27078, 'failed_unresolved': 0, 'reserved_for_batch': 0} selected: 27078
{'completed': 56, 'remaining': 27072, 'failed_unresolved': 0, 'reserved_for_batch': 0} | selected 6/27078 | elapsed 6.2s | 57.9 reviews-finished/min | selected ETA 28040s
{'completed': 57, 'remaining': 27071, 'failed_unresolved': 0, 'reserved_for_batch': 0} | selected 7/27078 | elapsed 61.2s | 6.9 reviews-finished/min | selected ETA 236627s
{'completed': 64, 'remaining': 27064, 'failed_unresolved': 0, 'reserved_for_batch': 0} | selected 14/27078 | elapsed 121.2s | 6.9 reviews-finished/min | selected ETA 234315s
{'completed': 70, 'remaining': 27058, 'failed_unresolved': 0, 'reserved_for_batch': 0} | selected 20/27078 | elapsed 181.2s | 6.6 reviews-finished/min | selected ETA 245120s
{'completed': 76, 'remaining': 27052, 'failed_unresolved': 0, 'reserved_for_batch': 0} | selected 26/27078 | elapsed 241.2s | 6.5 reviews-finished/min | selected ETA 250923s
{'completed': 83, 'rem

## B1 — Prepare Batch input and manifest (local only)
Creates a new folder with input JSONL and a manifest mapping each unique `custom_id` to its review/business IDs. Preparation does not upload or submit anything. It enforces the 50,000 request / 200 MB file limits; reduce RUN_LIMIT if needed. Also check your account's queued-token allowance before submitting.

To reopen an existing batch, set MANIFEST_PATH in C2 to its manifest and skip preparation. Do not prepare another batch just to check an existing one.


In [ ]:
PREPARE_BATCH = False
if PREPARE_BATCH:
    PREPARE_BATCH = False
    MANIFEST_PATH = prepare_batch(
        run, limit=RUN_LIMIT, retry_failed_only=RETRY_FAILED_ONLY,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
else:
    print('Set PREPARE_BATCH=True to prepare locally, or use your saved MANIFEST_PATH.')


## B2 — Upload and submit the prepared Batch (paid action)
This is the only Batch submission cell. A manifest can be submitted only once. Its state is saved **before** contacting the API, so an interruption cannot silently trigger resubmission. Do not automatically retry submission errors.


In [ ]:
SUBMIT_BATCH = False
if SUBMIT_BATCH:
    SUBMIT_BATCH = False
    assert MANIFEST_PATH, 'Choose or prepare a manifest first.'
    with OpenAI(max_retries=0, timeout=90.0) as batch_client:
        batch_id = submit_batch(run, batch_client, MANIFEST_PATH)
else:
    print('Not submitted. Set SUBMIT_BATCH=True only when ready.')


## B3 — Check status / recover a submission after restart (no labeling submission)
Rerun C1–C4 after restarting, select the existing MANIFEST_PATH, then run this cell. Checking status does not resubmit requests. If a submission was interrupted before its batch ID was saved, recovery searches server batches by manifest metadata/input file ID. Ambiguous/no matches stay blocked; inspect the dashboard rather than resubmitting.

Stopping the notebook does **not** stop a submitted server batch. You may cancel it in the dashboard, then collect its completed subset here. Keep unfinished batch IDs reserved until collection. To switch an unsubmitted prepared batch to concurrent mode, explicitly call `abandon_prepared(run, MANIFEST_PATH)`; it refuses submitted/uncertain batches.


In [ ]:
CHECK_BATCH_STATUS = False
if CHECK_BATCH_STATUS:
    CHECK_BATCH_STATUS = False
    assert MANIFEST_PATH, 'Set MANIFEST_PATH to the saved manifest.json.'
    with OpenAI(max_retries=0, timeout=90.0) as batch_client:
        manifest = json.loads(Path(MANIFEST_PATH).read_text())
        if not manifest.get('batch_id'):
            recover_batch(run, batch_client, MANIFEST_PATH)
        batch_status(run, batch_client, MANIFEST_PATH)
else:
    print('Set CHECK_BATCH_STATUS=True to check your selected batch.')


## B4 — Download, validate, and append Batch results
Run after completion, failure, expiry, or cancellation. Output ordering is ignored: unique custom IDs map responses back to reviews. Each successful response must pass ReviewLabels validation. Raw downloads are retained; refusals, invalid labels, non-200 responses, missing requests, and expiration errors go to the failure log.

Repeated collection never appends a successful review twice. After collection, unresolved failures become eligible for a **new explicitly prepared** retry batch (or concurrent mode). The old batch is never resubmitted.


In [ ]:
COLLECT_BATCH = False
if COLLECT_BATCH:
    COLLECT_BATCH = False
    assert MANIFEST_PATH, 'Set MANIFEST_PATH first.'
    with OpenAI(max_retries=0, timeout=90.0) as batch_client:
        collect_batch(run, batch_client, MANIFEST_PATH)
else:
    print('Set COLLECT_BATCH=True to retrieve your selected batch.')


## R1 — Progress and estimated usage cost (local only)
Counts distinguish successful reviews from unresolved failures. “Remaining” includes failed and batch-reserved reviews. Concurrent ETA applies to the selected run, uses measured throughput, and is approximate.

Usage includes available input/output/cached token counts from successes and failed responses. Old sequential records did not save usage; their cost is **unknown**, not zero. Retries or interrupted responses without returned usage are also excluded. Use the billing dashboard for actual charges. No fixed per-review price is assumed.


In [16]:
run.refresh()
print(run.counts())
usage = usage_report(run, PRICING_USD_PER_MILLION)


{'completed': 50, 'remaining': 27078, 'failed_unresolved': 0, 'reserved_for_batch': 0}
Estimated cost for recorded usage only (USD; not an invoice): {'input_tokens': 37521, 'cached_input_tokens': 0, 'output_tokens': 2609, 'estimated_usd': 0.0191828, 'records_without_usage': 0}
Excludes unknown usage from legacy records, interrupted requests, and retries without usage. Cached tokens are part of input, not extra input.


## R2 — Export current successes (local only, new timestamped file)
This retains review/business IDs, labels, model information, prompt version/hash/schema hash, dates, and available usage. Failed responses are never included.

In [17]:
run.refresh()
flat = [{**{k: v for k, v in r.items() if k != 'labels'}, **r['labels']}
        for r in run.success.values()]
df_labels = pd.DataFrame(flat)
export_path = RESULTS_FILE.with_name(
    RESULTS_FILE.stem + '_export_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f') + '.csv'
)
with export_path.open('x', encoding='utf-8', newline='') as file:
    df_labels.to_csv(file, index=False)
print('Exported:', len(df_labels), export_path)


Exported: 50 /Users/rejaimes/repos/coffee-project/data/processed/llm_labels_full_v3_8qualifiers_export_20261023_003209_723514.csv


## Prompt review — proposed changes only (not applied)
I did not edit `src/prompt_v3.txt`. During implementation it was updated externally to remove social_setting and request eight labels. The current normalized hash starts `00aad755`; the saved sequential results use `593ed4ed`. They cannot be mixed. The current prompt is used verbatim after the same `.strip()` normalization as before.

Proposed optional cleanup (not applied): consolidate the repeated beverage and work-friendliness rules. For example, the longer beverage rule already says food/pastry praise does not count; the shorter food-vs-drinks rule repeats it. Retain the explicit evidence and ambiguity rules. Wording changes could affect predictions and should be pilot-tested, not silently optimized.

**Changing any wording changes the prompt hash.** Use a new version/results file for a changed prompt; do not relabel historical records as compatible. Validate prompt changes on your pilot before a full run.

### Official references (checked 2026-10-22)
- [Batch guide](https://developers.openai.com/api/docs/guides/batch): 24h window, discounted pricing, unique custom IDs, output/error downloads, 50,000 requests / 200 MB, expiry behavior.
- [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs): strict JSON schema and refusal/incomplete handling. Both modes submit the same ReviewLabels schema and validate returned JSON with ReviewLabels locally.
- [Python SDK](https://developers.openai.com/api/reference/python): AsyncOpenAI, timeouts, configurable retries.
- [Rate limits](https://developers.openai.com/api/docs/guides/rate-limits): account limits and backoff.
- [GPT-4.1 mini](https://developers.openai.com/api/docs/models/gpt-4.1-mini): model capabilities and token pricing.

Local implementation: `src/labeling_runner.py`. Mock tests: `python3 -m unittest discover -s tests -p 'test_labeling_runner.py'`. Tests use fake clients and temporary files, never paid calls.
